In [1]:
import re
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional, Dict, List

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

In [2]:
NOTEBOOK_SUBDIR = "notebook 2"

ARTIFACTS = Path("artifacts")
IN_T2 = ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection"

OUT_CLEAN = ARTIFACTS / "cleaned" / NOTEBOOK_SUBDIR
OUT_META = ARTIFACTS / "metadata" / NOTEBOOK_SUBDIR
OUT_REPORTS = ARTIFACTS / "reports" / NOTEBOOK_SUBDIR

for d in [OUT_CLEAN, OUT_META, OUT_REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

# Assumptions consistent with your project spec
PRISM_PRIMARY_TAG = "lfc"
PRISM_SECONDARY_TAG = "auc"

# Expected inputs from Notebook 1 Track 2
INPUTS = {
    "rna": IN_T2 / "rna.parquet",
    "cnv": IN_T2 / "cnv.parquet",
    "mut": IN_T2 / "mut.parquet",
    "cell_metadata": IN_T2 / "cell_metadata.parquet",
    "prism_primary_pairs": IN_T2 / "prism_primary_pairs.parquet",
    "prism_secondary_pairs": IN_T2 / "prism_secondary_pairs.parquet",
    "prot_ms": IN_T2 / "prot_optional__prot_ms_ccle_gygi.parquet",
    "prot_rppa": IN_T2 / "prot_optional__prot_rppa_ccle.parquet",
    "prot_procan": IN_T2 / "prot_optional__prot_procan_depmapSanger.parquet",
    "prot_union": IN_T2 / "prot_optional__prot_combined_union.parquet",
}

missing = [k for k, p in INPUTS.items() if not p.exists() and "prot_" not in k]
if missing:
    print("[WARN] Missing required inputs:")
    for k in missing:
        print(" -", k, "->", INPUTS[k])
else:
    print("All required inputs found under:", IN_T2)


All required inputs found under: artifacts/aligned/notebook 1/track2_nonintersection


In [3]:
# Utility helpers

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def fingerprint(path: Path) -> dict:
    st = path.stat()
    return {
        "path": str(path.resolve()),
        "size_bytes": int(st.st_size),
        "mtime_utc": datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat(),
        "sha256": sha256_file(path),
    }

def write_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def read_parquet_strict(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet: {path}")
    df = pd.read_parquet(path)
    return df

def normalise_depmap_id(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip()

def normalise_compound_id(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip()

In [4]:
# Track 2 backbone

rna = read_parquet_strict(INPUTS["rna"])
cnv = read_parquet_strict(INPUTS["cnv"])
mut = read_parquet_strict(INPUTS["mut"])
cell_meta = read_parquet_strict(INPUTS["cell_metadata"])

# Ensure indices are clean strings
rna.index = normalise_depmap_id(pd.Series(rna.index, index=rna.index)).values
cnv.index = normalise_depmap_id(pd.Series(cnv.index, index=cnv.index)).values
mut.index = normalise_depmap_id(pd.Series(mut.index, index=mut.index)).values

# Metadata in Track 2 is saved as a table with depmap_id column (Notebook 1 used reset_index())
if "depmap_id" not in cell_meta.columns:
    # Fallback: if somehow depmap_id is the index
    cell_meta = cell_meta.reset_index().rename(columns={"index": "depmap_id"})
cell_meta["depmap_id"] = normalise_depmap_id(cell_meta["depmap_id"])

print("Loaded Track 2:")
print(" RNA:", rna.shape)
print(" CNV:", cnv.shape)
print(" MUT:", mut.shape)
print(" META:", cell_meta.shape)

# Define the Track 2 cohort as the intersection of the three backbone modalities
core_cells = sorted(set(rna.index) & set(cnv.index) & set(mut.index))
print("Core cohort (RNA+CNV+MUT):", len(core_cells))

Loaded Track 2:
 RNA: (1079, 19212)
 CNV: (1079, 19360)
 MUT: (1079, 19536)
 META: (1079, 7)
Core cohort (RNA+CNV+MUT): 1079


In [5]:
# Build cell_index.parquet

# One row per depmap_id in the Track 2 cohort
cell_index = pd.DataFrame({"depmap_id": core_cells})
cell_index = cell_index.merge(
    cell_meta.drop_duplicates("depmap_id"),
    on="depmap_id",
    how="left",
    validate="one_to_one",
)

# Basic hygiene: make lineage columns string if present
for c in list(cell_index.columns):
    if c.startswith("lineage_"):
        cell_index[c] = cell_index[c].astype("string")

# Helpful flags for downstream diagnostics
cell_index["in_track2_core"] = np.int8(1)

cell_index_path = OUT_CLEAN / "cell_index.parquet"
cell_index.to_parquet(cell_index_path, index=False)
print("Wrote:", cell_index_path, cell_index.shape)

# Save a small cohort summary for quick inspection
cohort_summary = {
    "n_core_cells": int(len(core_cells)),
    "n_meta_rows": int(cell_index.shape[0]),
    "n_unique_depmap_id": int(cell_index["depmap_id"].nunique()),
    "columns": list(map(str, cell_index.columns)),
}
write_json(cohort_summary, OUT_REPORTS / "cell_index_summary.json")

Wrote: artifacts/cleaned/notebook 2/cell_index.parquet (1079, 8)


In [6]:
# Build prism_long.parquet (standardised schema)

def load_prism_pairs(path: Path, target: str) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=["depmap_id", "compound_id", "y", "target"])

    df = pd.read_parquet(path)

    # Notebook 1 wrote columns depmap_id, compound, and value column lfc/auc
    df = df.copy()
    if "compound" in df.columns:
        df = df.rename(columns={"compound": "compound_id"})
    if "compound_id" not in df.columns:
        raise ValueError(f"PRISM pairs missing compound_id/compound column: {path}")

    # Identify response column
    value_cols = [c for c in df.columns if c not in ("depmap_id", "compound_id")]
    if len(value_cols) != 1:
        raise ValueError(f"Expected exactly one response column in {path}, got: {value_cols}")

    value_col = value_cols[0]
    df = df.rename(columns={value_col: "y"})

    df["depmap_id"] = normalise_depmap_id(df["depmap_id"])
    df["compound_id"] = normalise_compound_id(df["compound_id"])
    df["y"] = pd.to_numeric(df["y"], errors="coerce").astype("float32")
    df["target"] = target

    # Drop any rows with missing y after coercion
    df = df.dropna(subset=["y"])

    # Optional: restrict to the frozen Track 2 cohort (should already match, but keeps it explicit)
    df = df[df["depmap_id"].isin(core_cells)]

    return df[["depmap_id", "compound_id", "y", "target"]]

prism_primary = load_prism_pairs(INPUTS["prism_primary_pairs"], target=PRISM_PRIMARY_TAG)
prism_secondary = load_prism_pairs(INPUTS["prism_secondary_pairs"], target=PRISM_SECONDARY_TAG)

prism_long = pd.concat([prism_primary, prism_secondary], ignore_index=True)

# Add stable descriptive columns that help later notebooks avoid re-encoding assumptions
target_meta = {
    PRISM_PRIMARY_TAG: {
        "dataset": "PRISM",
        "screen": "Repurposing_Primary",
        "metric": "LFC_vs_DMSO",
        "task": "regression",
        "direction": "lower_is_more_sensitive",
    },
    PRISM_SECONDARY_TAG: {
        "dataset": "PRISM",
        "screen": "Repurposing_Secondary",
        "metric": "AUC",
        "task": "regression",
        "direction": "lower_is_more_sensitive",
    },
}
prism_long["screen"] = prism_long["target"].map(lambda t: target_meta.get(t, {}).get("screen", "Unknown")).astype("string")
prism_long["metric"] = prism_long["target"].map(lambda t: target_meta.get(t, {}).get("metric", "Unknown")).astype("string")

# Sort for reproducibility
prism_long = prism_long.sort_values(["target", "compound_id", "depmap_id"]).reset_index(drop=True)

prism_long_path = OUT_CLEAN / "prism_long.parquet"
prism_long.to_parquet(prism_long_path, index=False)
print("Wrote:", prism_long_path, prism_long.shape)

# Quick coverage summaries
cov_by_target = (
    prism_long.groupby("target")
    .agg(
        n_pairs=("y", "size"),
        n_cells=("depmap_id", "nunique"),
        n_compounds=("compound_id", "nunique"),
    )
    .reset_index()
)
cov_by_target.to_csv(OUT_REPORTS / "prism_long__coverage_by_target.csv", index=False)

Wrote: artifacts/cleaned/notebook 2/prism_long.parquet (3419786, 6)


In [7]:
# Build gene_index.parquet (feature-to-gene mapping)
# -------------------------
# Canonical gene key choice:
# - We keep gene_symbol_canonical as the primary interpretability key.
# - Where a feature is not cleanly a gene symbol, we use best-effort parsing and keep the raw feature too.

def strip_platform_prefix(feature: str) -> (str, Optional[str]):
    """
    For namespaced union features like "ms__TP53", returns ("TP53", "ms").
    If not namespaced, returns (feature, None).
    """
    s = str(feature)
    if "__" in s:
        pref, rest = s.split("__", 1)
        return rest, pref
    return s, None

_gene_token_re = re.compile(r"^[A-Za-z0-9]+(?:[-_][A-Za-z0-9]+)*$")

def canonicalise_gene_symbol(feature: str, modality: str) -> str:
    """
    Best-effort normalisation into a gene-like token.
    This is intentionally conservative: it does not try to be a full HGNC resolver.
    """
    s = str(feature).strip()

    # Remove namespacing for union features first
    if modality == "prot_combined_union":
        s, _ = strip_platform_prefix(s)

    # Common clean-up
    s = s.replace("(", "").replace(")", "").strip()

    # RPPA tends to include antibody or phospho labels; take the first token as a baseline
    if modality in ("prot_rppa_ccle", "rppa"):
        # Example patterns often look like "TP53_pS15" or "AKT1" etc
        s = re.split(r"[ _|:;]", s)[0].strip()

    # ProCan can include IDs or composite strings; take the first token before separators
    if modality in ("prot_procan_depmapSanger", "procan"):
        s = re.split(r"[|:; ]", s)[0].strip()

    # Mut/CNV/RNA/MS are typically already gene symbols; still sanitise
    s = re.split(r"[|:; ]", s)[0].strip()

    # Upper-case canonical gene symbol tokens for consistency (common in DepMap tables)
    s_up = s.upper()

    # Only accept "gene-like" tokens, otherwise keep a safe fallback based on the raw token
    if _gene_token_re.match(s_up):
        return s_up

    # Fallback: remove non-alphanumerics and upper-case
    s_fallback = re.sub(r"[^A-Za-z0-9_-]", "", s_up).strip("_-")
    return s_fallback if s_fallback else s_up

def build_feature_rows(features: List[str], modality: str, source: str) -> pd.DataFrame:
    rows = []
    for f in features:
        raw = str(f)
        canonical = canonicalise_gene_symbol(raw, modality=modality)
        base, prefix = strip_platform_prefix(raw) if modality == "prot_combined_union" else (raw, None)
        rows.append({
            "feature_name": raw,
            "feature_base": str(base),
            "platform_prefix": str(prefix) if prefix is not None else pd.NA,
            "modality": modality,
            "source": source,
            "gene_symbol_canonical": canonical,
        })
    return pd.DataFrame(rows)

# Collect features from backbone modalities
gene_rows = []
gene_rows.append(build_feature_rows(list(map(str, rna.columns)), modality="rna", source="depmap_rna"))
gene_rows.append(build_feature_rows(list(map(str, cnv.columns)), modality="cnv", source="depmap_cnv"))
gene_rows.append(build_feature_rows(list(map(str, mut.columns)), modality="mut", source="depmap_mut"))

# Optional: include proteomics feature sets from Track 2 optional matrices
def maybe_load_prot_features(path: Path) -> List[str]:
    if not path.exists():
        return []
    df = pd.read_parquet(path)
    return list(map(str, df.columns))

prot_arms = {
    "prot_ms_ccle_gygi": INPUTS["prot_ms"],
    "prot_rppa_ccle": INPUTS["prot_rppa"],
    "prot_procan_depmapSanger": INPUTS["prot_procan"],
    "prot_combined_union": INPUTS["prot_union"],
}

for arm, p in prot_arms.items():
    feats = maybe_load_prot_features(p)
    if feats:
        gene_rows.append(build_feature_rows(feats, modality=arm, source=f"track2_prot_optional::{arm}"))
        print(f"Included proteomics features for {arm}: {len(feats)}")
    else:
        print(f"[INFO] Proteomics features not included for {arm} (file missing or empty).")

gene_index = pd.concat(gene_rows, ignore_index=True)

# De-duplicate exact (modality, feature_name) duplicates, keeping first (should be rare)
gene_index = gene_index.drop_duplicates(subset=["modality", "feature_name"]).reset_index(drop=True)

# A small mapping quality report: how often canonical differs from raw base
gene_index["canonical_changed"] = (gene_index["gene_symbol_canonical"].astype(str) != gene_index["feature_base"].astype(str)).astype(np.int8)
map_report = (
    gene_index.groupby("modality")
    .agg(
        n_features=("feature_name", "size"),
        n_changed=("canonical_changed", "sum"),
        frac_changed=("canonical_changed", "mean"),
        n_unique_genes=("gene_symbol_canonical", "nunique"),
    )
    .reset_index()
)
map_report.to_csv(OUT_REPORTS / "gene_index__mapping_report.csv", index=False)

gene_index_path = OUT_CLEAN / "gene_index.parquet"
gene_index.to_parquet(gene_index_path, index=False)
print("Wrote:", gene_index_path, gene_index.shape)

Included proteomics features for prot_ms_ccle_gygi: 11780
Included proteomics features for prot_rppa_ccle: 144
Included proteomics features for prot_procan_depmapSanger: 7906
Included proteomics features for prot_combined_union: 18751
Wrote: artifacts/cleaned/notebook 2/gene_index.parquet (96689, 7)


In [8]:
# Validation checks (lightweight, but catches silent join errors)

# 1) PRISM depmap_id coverage
if prism_long.shape[0] > 0:
    pct_mapped = prism_long["depmap_id"].isin(set(cell_index["depmap_id"])).mean() * 100.0
    print(f"PRISM long depmap_id mapping into cell_index: {pct_mapped:.2f}%")

# 2) Cohort consistency across modalities
assert set(core_cells) == set(cell_index["depmap_id"]), "cell_index cohort mismatch"
assert set(core_cells) == set(rna.index) & set(cnv.index) & set(mut.index), "core cohort definition mismatch"

# 3) Basic schema expectations
required_cell_cols = {"depmap_id"}
required_prism_cols = {"depmap_id", "compound_id", "y", "target"}
required_gene_cols = {"feature_name", "modality", "gene_symbol_canonical"}

assert required_cell_cols.issubset(cell_index.columns), "cell_index missing required columns"
assert required_prism_cols.issubset(prism_long.columns), "prism_long missing required columns"
assert required_gene_cols.issubset(gene_index.columns), "gene_index missing required columns"

print("Validation checks passed.")

PRISM long depmap_id mapping into cell_index: 100.00%
Validation checks passed.


In [9]:
# Notebook 2 lock file (freeze key outputs + fingerprints)

lock = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": NOTEBOOK_SUBDIR,
    "inputs": {},
    "outputs": {},
    "notes": {
        "canonical_cell_id": "depmap_id",
        "canonical_gene_key": "gene_symbol_canonical",
        "prism_long_schema": ["depmap_id", "compound_id", "y", "target", "screen", "metric"],
        "track2_backbone": "RNA+CNV+MUT cohort from artifacts/aligned/notebook 1/track2_nonintersection",
    },
}

# Fingerprint required inputs (skip missing optional proteomics gracefully)
for k, p in INPUTS.items():
    if p.exists():
        lock["inputs"][k] = fingerprint(p)
    else:
        lock["inputs"][k] = {"path": str(p), "exists": False}

# Fingerprint outputs
for k, p in {
    "cell_index": cell_index_path,
    "gene_index": gene_index_path,
    "prism_long": prism_long_path,
    "mapping_report_csv": OUT_REPORTS / "gene_index__mapping_report.csv",
    "prism_coverage_csv": OUT_REPORTS / "prism_long__coverage_by_target.csv",
}.items():
    if p.exists():
        lock["outputs"][k] = fingerprint(p)

lock_path = OUT_META / "notebook2_lock.json"
write_json(lock, lock_path)
print("Wrote:", lock_path)

print("\nCoverage by target:")
print(cov_by_target)

print("\nGene mapping report (head):")
print(map_report.head(12))

Wrote: artifacts/metadata/notebook 2/notebook2_lock.json

Coverage by target:
  target  n_pairs  n_cells  n_compounds
0    auc   306903      377         1450
1    lfc  3112883      617         6767

Gene mapping report (head):
                   modality  n_features  n_changed  frac_changed  n_unique_genes
0                       cnv       19360        229      0.011829           19360
1                       mut       19536        209      0.010698           19536
2       prot_combined_union       18751        272      0.014506           11470
3         prot_ms_ccle_gygi       11780         77      0.006537           11780
4  prot_procan_depmapSanger        7906         61      0.007716            7906
5            prot_rppa_ccle         144        144      1.000000             144
6                       rna       19212        224      0.011659           19212


# Notebook 02: Harmonisation and Join Index Construction 

## Purpose
This notebook standardises the project’s *join backbone* and response schema for downstream modelling. It converts the Track 2 outputs from Notebook 1 into three canonical artefacts: a cell-level index, a long-form PRISM response table, and a feature-to-gene mapping table. The goal is to ensure consistent identifiers, reproducible joins, and explicit target metadata before any modelling or evaluation.

## Inputs
Loaded from `artifacts/aligned/notebook 1/track2_nonintersection/`:

- Backbone omics: `rna.parquet`, `cnv.parquet`, `mut.parquet`
- Cell metadata: `cell_metadata.parquet`
- PRISM response pairs: `prism_primary_pairs.parquet`, `prism_secondary_pairs.parquet`
- Optional proteomics per arm: `prot_optional__<arm>.parquet` for MS, RPPA, ProCan, and combined union

## Core definitions
The canonical cell identifier is `depmap_id`. Track 2 is defined as the largest cohort with complete backbone modalities (RNA + CNV + MUT). Drug response is represented as observed `(depmap_id, compound_id)` pairs and stored in a unified long table, with the primary target set to PRISM LFC and the secondary target set to PRISM AUC.

## Outputs
Written to `artifacts/cleaned/notebook 2/` and metadata to `artifacts/metadata/notebook 2/`:

## 1) Cell index
`cell_index.parquet` contains one row per `depmap_id` in the Track 2 backbone cohort, merged with available lineage metadata and basic cohort flags. This table is the single source of truth for cell-line joins across all later notebooks.

### 2) PRISM long table
`prism_long.parquet` stores PRISM responses in a standardised schema:
`(depmap_id, compound_id, y, target, screen, metric)`.
Targets include `lfc` (primary) and `auc` (secondary). Values are coerced to numeric and filtered to the Track 2 cohort to prevent silent join drift.

## 3) Gene index
`gene_index.parquet` provides a feature inventory and best-effort mapping from raw feature names to a canonical gene-like token for interpretability. It includes modality provenance, optional platform prefix extraction for namespaced union features, and a mapping quality report (`gene_index__mapping_report.csv`) to quantify how often canonicalisation alters raw tokens.

## 4) Reproducibility lock
`notebook2_lock.json` fingerprints all inputs and key outputs (paths, sizes, mtimes, SHA-256) to freeze the harmonisation state and enable exact re-runs.

## Quality checks
- Enforced cohort consistency: Track 2 core cohort equals the intersection of RNA, CNV, and MUT indices.
- Verified response join integrity: PRISM `depmap_id` coverage in `cell_index` is reported and expected to be near-complete for aligned inputs.
- Validated schemas: required columns exist for `cell_index`, `prism_long`, and `gene_index`.

## Notes for downstream notebooks
`cell_index.parquet` and `prism_long.parquet` should be treated as immutable backbone artefacts. Any preprocessing choices that learn statistics (imputation, scaling, PCA) must be applied fold-wise in later notebooks to remain leakage-safe. The gene index is intended for reporting and interpretability; feature identity should remain based on raw feature names unless an explicit aggregation policy is introduced.
